# Проект по эконометрике 

In [75]:
import pandas as pd
import numpy as np

In [119]:
df = pd.read_csv('all_data_final_llm.csv')
df = df.rename(columns={'index':'rang_id'})
df

,has_secondary_education,has_higher_education,work_experience,master_id,master_id.1,rang_id,docs_verified,gender,guarantee,rating,...,is_certified,price_socket,suitabilityHints,discount,insurance,contract,cur_score,profi_ru_start,album_count,photos_count
0,True,False,7.0,https://profi.ru/profile/AbdullinAA22/,https://profi.ru/profile/AbdullinAA22/,110.0,True,NaN,Нет гарантии,0,...,False,NaN,NaN,NaN,NaN,NaN,50.679234,На сервисе с июня 2025 г,0,0
1,True,False,8.0,https://profi.ru/profile/AbdullinAM9/,https://profi.ru/profile/AbdullinAM9/,158.0,True,MALE,Нет гарантии,0,...,False,NaN,NaN,NaN,NaN,NaN,50.700671,На сервисе с ноября 2024 г. (1 год),0,7
2,True,False,6.0,https://profi.ru/profile/AbdullinAN2/,https://profi.ru/profile/AbdullinAN2/,131.0,True,MALE,По договоренности,"4,73",...,False,NaN,Мало отзывов по вашей услуге,NaN,NaN,NaN,50.761180,На сервисе с октября 2020 г. (5 лет),0,2
3,False,True,NaN,https://profi.ru/profile/AbdullinLR/,https://profi.ru/profile/AbdullinLR/,39.0,True,MALE,Нет гарантии,"5,0",...,False,NaN,NaN,NaN,NaN,NaN,50.500441,NaN,0,0
4,True,False,7.0,https://profi.ru/profile/AbrashevYI/,https://profi.ru/profile/AbrashevYI/,85.0,True,MALE,Нет гарантии,0,...,False,NaN,NaN,NaN,NaN,NaN,50.782032,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
738,False,False,23.0,https://profi.ru/profile/ZinnurovDI2/,https://profi.ru/profile/ZinnurovDI2/,108.0,True,MALE,Нет гарантии,0,...,False,NaN,NaN,NaN,NaN,NaN,50.539320,NaN,2,3
739,True,False,1.0,https://profi.ru/profile/ZiyautdinovFZ/,https://profi.ru/profile/ZiyautdinovFZ/,95.0,True,MALE,Нет гарантии,0,...,False,NaN,NaN,NaN,NaN,NaN,50.787963,На сервисе с марта 2025 г. (1 год),0,0
740,True,False,7.0,https://profi.ru/profile/ZlobinAA22/,https://profi.ru/profile/ZlobinAA22/,48.0,True,MALE,Нет гарантии,0,...,False,NaN,NaN,NaN,NaN,NaN,50.798721,На сервисе с сентября 2025 г,0,0
741,True,False,10.0,https://profi.ru/profile/ZvitseviLA/,https://profi.ru/profile/ZvitseviLA/,76.0,True,MALE,Нет гарантии,"4,76",...,False,NaN,NaN,NaN,NaN,NaN,50.753450,NaN,0,0


In [120]:
df.isna().sum()

has_secondary_education     59
has_higher_education        59
work_experience            128
master_id                    0
master_id.1                  0
rang_id                      0
docs_verified                0
gender                      69
guarantee                    0
rating                       0
reviews_count                0
is_highly_praised            0
is_certified                 0
price_socket               646
suitabilityHints           610
discount                   650
insurance                  737
contract                   706
cur_score                    0
profi_ru_start             324
album_count                  0
photos_count                 0
dtype: int64

In [121]:
def guarantee_transform(value):
## переводим гарантию в числа в месяцах

    ## без гарантии
    if value in ['Нет гарантии', 'По договоренности', '«Штукатурка и изделия из ГКЛ, ГВЛ – 1 год»', '«Гарантия это надежность выполненных работ»']:
        return 0
    
    ## 1 год
    elif value in ['«1 год»', '«Гарантия на работы 1 год»', '«На 1 год»', '«Гарантия на выполненную работу 12 месяцев»',
                              '«В течении года при условии правильного пользования (если клиент не виноват в проблеме) + сервисное обслуживание на льготных условиях (по договоренности)»']:
        return 12
    
    ## 6 месяцев
    elif value in ['«6 месяцев»', '«От 6 месяцев до 1 года»', '«Гарантия 6 месяцев с окончания работ»']:
        return 6
    
    ## 2 года
    elif value in ['«2 года»', '«2 года, при условии подбора материалов мастером и соблюдении технологии работ»', '«Гарантия 2 года с техническим сопровождением»']:
        return 24
    
    ## 3 года
    elif value in ['«3 года»']:
        return 36
    
    ## 1 месяц
    elif value in ['«Гарантия на Услуги от 30 дней»', '«1 месяц»', 
                              '«В разумных пределах на то что от меня зависит. Можно прописывать в Договоре»', 
                              '«Даю гарантию на выполненную работу»', 'По договоренности', 
                              '«Работаем с договором »', '«От 14 до 400 дней»', 
                              '«Договор подряда, и гарантия на работы до 10 лет»']:
        return 1
    
    return None


months = {'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4, 'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8, 'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12}
def profiru_start_transformer(value):
    month_count = None
    if pd.notna(value):
        list_val = value.split()
        month = list_val[3]
        year = int(list_val[4])
        month_count = 12 * (2026 - year) + (5 - months[month])
    
    return month_count


df['no_electric_reviews'] = None
df['few_electric_reviews'] = None

def suitabilityHints_transformer(row):
    if row['suitabilityHints'] == 'Нет отзывов по вашей услуге':
        row['no_electric_reviews'] = 1

    elif row['suitabilityHints'] == 'Мало отзывов по вашей услуге':
        row['few_electric_reviews'] = 1

    return row


df['guarantee'] = df['guarantee'].apply(guarantee_transform)
df['is_highly_praised'] = df['is_highly_praised'].apply(lambda x: 1 if x==True else 0)
df['is_certified'] = df['is_certified'].apply(lambda x: 1 if x==True else 0)

df['docs_verified'] = df['docs_verified'].apply(lambda x: 1 if x==True else 0)
df.drop(columns=['gender'], inplace=True)

df['has_secondary_education'] = df['has_secondary_education'].apply(lambda x: 1 if x==True else 0)
df['has_higher_education'] = df['has_higher_education'].apply(lambda x: 1 if x==True else 0)

df.drop(columns=['master_id.1'], inplace=True)

df['has_contract'] = df['contract'].apply(lambda x: 1 if x=='Договор' else 0)
df.drop(columns=['contract'], inplace=True)

df['has_insurance'] = df['insurance'].apply(lambda x: 1 if x=='Договор' else 0)
df.drop(columns=['insurance'], inplace=True)

df['months_on_profi_ru'] = df['profi_ru_start'].apply(profiru_start_transformer)
df.drop(columns=['profi_ru_start'], inplace=True)

df.drop(columns=['is_certified'], inplace=True)
df.drop(columns=['is_highly_praised'], inplace=True)

df = df.apply(suitabilityHints_transformer, axis=1)
df.drop(columns=['suitabilityHints'], inplace=True)

df['no_electric_reviews'] = df['no_electric_reviews'].fillna(0)
df['few_electric_reviews'] = df['few_electric_reviews'].fillna(0)


df

,has_secondary_education,has_higher_education,work_experience,master_id,rang_id,docs_verified,guarantee,rating,reviews_count,price_socket,discount,cur_score,album_count,photos_count,no_electric_reviews,few_electric_reviews,has_contract,has_insurance,months_on_profi_ru
0,1,0,7.0,https://profi.ru/profile/AbdullinAA22/,110.0,1,0,0,0,NaN,NaN,50.679234,0,0,0.0,0.0,0,0,11.0
1,1,0,8.0,https://profi.ru/profile/AbdullinAM9/,158.0,1,0,0,0,NaN,NaN,50.700671,0,7,0.0,0.0,0,0,18.0
2,1,0,6.0,https://profi.ru/profile/AbdullinAN2/,131.0,1,0,"4,73",15,NaN,NaN,50.761180,0,2,0.0,1.0,0,0,67.0
3,0,1,NaN,https://profi.ru/profile/AbdullinLR/,39.0,1,0,"5,0",3,NaN,NaN,50.500441,0,0,0.0,0.0,0,0,NaN
4,1,0,7.0,https://profi.ru/profile/AbrashevYI/,85.0,1,0,0,0,NaN,NaN,50.782032,0,0,0.0,0.0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
738,0,0,23.0,https://profi.ru/profile/ZinnurovDI2/,108.0,1,0,0,0,NaN,NaN,50.539320,2,3,0.0,0.0,0,0,NaN
739,1,0,1.0,https://profi.ru/profile/ZiyautdinovFZ/,95.0,1,0,0,0,NaN,NaN,50.787963,0,0,0.0,0.0,0,0,14.0
740,1,0,7.0,https://profi.ru/profile/ZlobinAA22/,48.0,1,0,0,0,NaN,NaN,50.798721,0,0,0.0,0.0,0,0,8.0
741,1,0,10.0,https://profi.ru/profile/ZvitseviLA/,76.0,1,0,"4,76",68,NaN,NaN,50.753450,0,0,0.0,0.0,0,0,NaN


In [114]:
df.columns

Index(['has_secondary_education', 'has_higher_education', 'work_experience',
       'master_id', 'rang_id', 'docs_verified', 'guarantee', 'rating',
       'reviews_count', 'price_socket', 'discount', 'cur_score', 'album_count',
       'photos_count', 'no_electric_reviews', 'few_electric_reviews',
       'has_contract', 'has_insurance', 'months_on_profi_ru'],
      dtype='object')

In [122]:
df.to_csv('processed_data_kazan.csv', index=False)

- 'has_secondary_education' - закончил училище, ПТУ, колледж и т.д.
- 'has_higher_education' - закончил вуз
- 'work_experience' - число лет опыта
- 'master_id' - ссылка на профиль
-'rang_id' - место в выдаче
- 'docs_verified' - документы подтверждены профи ру
- 'guarantee' - дает гарантию на работу, в месяцах
- 'rating' - рейтинг по отзывам (звездочки из 5 в профиле)
- 'reviews_count' - число отзывов
- 'price_socket' - цена установки розетки
- 'discount' - процент скидки, которую дает (иногда бывает только для определенных категорий людей)
- 'cur_score' - оценка по скоринговой модели с Профи ру
- 'album_count' - число альбомов с фотографиями с работой в профиле
-  'photos_count' - число фото с работой вне альбомов в профиле
- 'no_electric_reviews'- лейбл на сайте, что нет отзывов по услуге установки розетки
- 'few_electric_reviews' - лейбл на сайте, что мало отзывов по услуге установки розетки
- 'has_contract' - предлагает работу по договору
- 'has_insurance' - застраховал свою работу на Профи ру (если не понравится работа, можно обратиться на Профи и вернуть деньги, т.к. мастер застраховал это)
- 'months_on_profi_ru' - сколько месяцев человек на профи ру